In [ ]:
# =============================================================
# PEAP on LEMON Mu/Alpha — Kirchhoff et al. 2026 (FIR GT only)
# =============================================================
# Broadband Burg AR -> extrapolate -> acausal FIR -> Hilbert
# Scored against a single zero-phase FIR ground truth.
#
# ── OUTPUT SCHEMA MATCHED 1:1 TO THE EPN NOTEBOOK ────────────
# PEAP uses fixed paper params (no per-subject training) -> n_train_samples = 0.

import numpy as np
import time, gc, os, json
from scipy.signal import filtfilt, hilbert, firwin, resample, welch
from scipy.stats import pearsonr, spearmanr

# =============================================================
# Load preprocessed subjects from the .npz cache
# =============================================================
LOAD_DIR = '/content/drive/MyDrive/RAIN_IRC/LEMON'
OUTDIR   = '/content/drive/MyDrive/RAIN_IRC/result_alpha/PEAP_lemon'

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f"(Skipping Drive mount: {e})")

os.makedirs(OUTDIR, exist_ok=True)

all_subjects = {}
for fname in sorted(os.listdir(LOAD_DIR)):
    if not fname.endswith('.npz'):
        continue
    subj_id = fname[:-4]
    d = np.load(os.path.join(LOAD_DIR, fname), allow_pickle=True)
    all_subjects[subj_id] = {
        'epochs':   d['epochs'],
        'fs':       float(d['fs']),
        'n_epochs': int(d['n_epochs']),
        'cal_n':    int(d['cal_n']),
        'test_n':   int(d['test_n']),
        'montage':  str(d['montage']) if 'montage' in d.files else 'C3-Hjorth',
    }
    d.close()

if not all_subjects:
    raise FileNotFoundError(f"No .npz files found in {LOAD_DIR}")

FS = float(next(iter(all_subjects.values()))['fs'])
print(f"Loaded {len(all_subjects)} subjects from {LOAD_DIR} @ {FS:.0f} Hz\n")


# --- PEAP parameters (from paper, Table: optimal hyperparameters) ---
PEAP_FS = 1000.0
PEAP_AR_ORDER = 130
PEAP_CONTEXT_MS = 980
PEAP_PAD_MS = 290
PEAP_CONTEXT_SAMP = int(PEAP_CONTEXT_MS * PEAP_FS / 1000)  # 980
PEAP_PAD_SAMP = int(PEAP_PAD_MS * PEAP_FS / 1000)          # 290  # FIX: was mislabeled 130 in comment
PEAP_FIR_BAND = [9.0, 13.0]                                # passband (Hz)

# FIX: TWO decoupled filters. The GT order is the experimental variable;
#      PEAP's internal filter is FIXED at the paper config and never swept.
GT_FIR_ORDER            = 230   # <-- sweep this (scoring reference only)
PEAP_INTERNAL_FIR_ORDER = 230    # <-- paper config, DO NOT sweep (method's own filter)

# Eval config
EVAL_TIMES_S = [1.5, 2.0, 2.5, 3.0, 3.5]
EVAL_TIMES_SAMP_1K     = [int(t * PEAP_FS) for t in EVAL_TIMES_S]
PRE_SAMP               = 0
EVAL_TIMES_SAMP_NATIVE = [PRE_SAMP + int(t * FS) for t in EVAL_TIMES_S]

# Descriptor filter: FIXED 231-tap, kept cross-comparable with EPN regardless of GT sweep.
DESC_FIR_ORDER = 230
ALPHA_SEARCH_LO, ALPHA_SEARCH_HI = 7.0, 14.0

# --- Build the three filter sets ---
# GT filter (scoring reference)
gt_b = firwin(GT_FIR_ORDER + 1, PEAP_FIR_BAND, fs=PEAP_FS,
              window='hamming', pass_zero=False)
gt_a = np.array([1.0])

# PEAP internal filter (fixed paper config)
peap_b = firwin(PEAP_INTERNAL_FIR_ORDER + 1, PEAP_FIR_BAND, fs=PEAP_FS,
                window='hamming', pass_zero=False)
peap_a = np.array([1.0])

# Descriptor filter (fixed 231-tap, EPN-comparable)
desc_b = firwin(DESC_FIR_ORDER + 1, PEAP_FIR_BAND, fs=PEAP_FS,
                window='hamming', pass_zero=False)
desc_a = np.array([1.0])

print(f"GT filter      : {len(gt_b)} taps (order {GT_FIR_ORDER}, filtfilt) -- SCORING REFERENCE")
print(f"PEAP internal  : {len(peap_b)} taps (order {PEAP_INTERNAL_FIR_ORDER}) -- fixed paper config")
print(f"Descriptor     : {len(desc_b)} taps (order {DESC_FIR_ORDER}) -- EPN-comparable ITPC")
print(f"PEAP AR: Burg order {PEAP_AR_ORDER} | context {PEAP_CONTEXT_MS}ms | pad {PEAP_PAD_MS}ms")
print(f"Resampling: {FS:.0f} Hz -> {PEAP_FS:.0f} Hz")

# FIX: GT at order 1460 needs epochs longer than ~4.38 s or filtfilt degrades.
GT_MIN_LEN = 3 * len(gt_b) + 1
print(f"GT requires epoch length > {GT_MIN_LEN} samples ({GT_MIN_LEN/PEAP_FS:.2f}s) to run cleanly")


# --- Burg AR (unchanged) ---
def burg_ar(x, order):
    x = x.astype(np.float64); n = len(x); order = int(order)
    if n <= order + 1: return np.zeros(order, dtype=np.float64)
    ef, eb = x.copy(), x.copy(); a = np.zeros(order, dtype=np.float64)
    for m in range(order):
        ef_s, eb_s = ef[m+1:n], eb[m:n-1]; L = min(len(ef_s), len(eb_s))
        if L <= 0: break
        ef_s, eb_s = ef_s[:L].copy(), eb_s[:L].copy()
        num = -2.0 * np.dot(ef_s, eb_s); den = np.dot(ef_s, ef_s) + np.dot(eb_s, eb_s)
        if den < 1e-12: break
        k = np.clip(num / den, -0.9999, 0.9999)
        a_new = np.zeros(m + 1, dtype=np.float64); a_new[m] = k
        if m > 0: a_new[:m] = a[:m] + k * a[m-1::-1]
        a[:m + 1] = a_new
        ef[m+1:m+1+L] = ef_s + k * eb_s; eb[m+1:m+1+L] = eb_s + k * ef_s; n = m + 1 + L
    return -a

def ar_extrapolate(x, coeffs, n_steps):
    p = len(coeffs)
    buf = list(x.astype(np.float64))
    x_max = np.max(np.abs(x)) * 10.0
    for _ in range(n_steps):
        if len(buf) < p: buf.append(buf[-1]); continue
        pred = sum(coeffs[k] * buf[-k-1] for k in range(p))
        if abs(pred) > x_max: break
        buf.append(pred)
    return np.array(buf, dtype=np.float64)


# --- PEAP phase estimation ---
def peap_predict_phase(raw_signal_1k, eval_samp):
    """
    PEAP pipeline (Kirchhoff et al. 2026), using the FIXED internal filter (peap_b).
    Returns (sin phi, cos phi), or (nan, nan) on failure.  # FIX: NaN, not (0,1)
    """
    ctx_start = max(0, eval_samp - PEAP_CONTEXT_SAMP + 1)
    context = raw_signal_1k[ctx_start:eval_samp + 1].astype(np.float64)
    ctx_len = len(context)

    if ctx_len < PEAP_AR_ORDER + 5 or np.std(context) < 1e-10:
        return np.nan, np.nan  # FIX

    ar_order = min(PEAP_AR_ORDER, ctx_len // 3)
    coeffs = burg_ar(context, ar_order)
    padded = ar_extrapolate(context, coeffs, PEAP_PAD_SAMP)

    # FIX: guard against PEAP's OWN filter (peap_b, 231 taps -> needs >694 samples).
    #      padded is ~1270 samples, so this now passes instead of always short-circuiting.
    min_filtfilt_len = 3 * len(peap_b) + 1
    if len(padded) < min_filtfilt_len:
        return np.nan, np.nan  # FIX

    try:
        filtered = filtfilt(peap_b, peap_a, padded,          # FIX: peap_b, not the GT filter
                            padlen=min(3 * len(peap_b), len(padded) - 1))
    except Exception:
        return np.nan, np.nan  # FIX

    analytic = hilbert(filtered)
    target_idx = ctx_len - 1
    if target_idx >= len(analytic):
        return np.nan, np.nan  # FIX

    ph = np.angle(analytic[target_idx])
    return np.sin(ph), np.cos(ph)


# --- GT computation at 1000 Hz (uses the SWEPT gt_b) ---
def compute_gt_fir(epoch_1k):
    """GT: zero-phase FIR order GT_FIR_ORDER (filtfilt) + Hilbert.
    Returns NaN arrays if the epoch is too short for the GT filter."""  # FIX
    if len(epoch_1k) < GT_MIN_LEN:
        n = len(epoch_1k)
        return np.full(n, np.nan, np.float32), np.full(n, np.nan, np.float32)  # FIX
    f = filtfilt(gt_b, gt_a, epoch_1k.astype(np.float64),
                padlen=min(3 * len(gt_b), len(epoch_1k) - 1))
    phase = np.angle(hilbert(f))
    return np.sin(phase).astype(np.float32), np.cos(phase).astype(np.float32)


# --- Descriptor GT (FIXED 231-tap, EPN-comparable ITPC) ---
def compute_desc_fir(epoch_1k):  # FIX: separate fixed filter so ITPC stays joinable with EPN
    min_len = 3 * len(desc_b) + 1
    if len(epoch_1k) < min_len:
        n = len(epoch_1k)
        return np.full(n, np.nan, np.float32), np.full(n, np.nan, np.float32)
    f = filtfilt(desc_b, desc_a, epoch_1k.astype(np.float64),
                padlen=min(3 * len(desc_b), len(epoch_1k) - 1))
    phase = np.angle(hilbert(f))
    return np.sin(phase).astype(np.float32), np.cos(phase).astype(np.float32)


# =============================================================
# Metrics + alpha descriptors  (EPN/BOB-IDENTICAL)
# =============================================================
def errors_to_metrics(errors):
    errors = np.asarray(errors, dtype=np.float64)
    errors = errors[np.isfinite(errors)]  # FIX: drop NaNs before scoring
    if len(errors) < 10:
        return {'mace': 90.0, 'plv': 0.0, 'csd': 180.0, 'acc': 0.0, 'bias': 0.0, 'n_samples': 0}
    mean_abs = float(np.mean(np.abs(errors))); R = float(np.abs(np.mean(np.exp(1j * errors))))
    return {'mace': float(np.degrees(mean_abs)), 'plv': R,
            'csd': float(np.degrees(np.sqrt(-2.0 * np.log(max(R, 1e-12))))),
            'acc': float((1.0 - mean_abs / np.pi) * 100.0),
            'bias': float(np.degrees(np.angle(np.mean(np.exp(1j * errors))))),
            'n_samples': int(len(errors))}

def itpc_from_gt(gt, trials, esamp):
    """GT-phase ITPC across trials at one eval sample."""
    ph = []
    for tr in trials:
        if tr in gt:
            gs, gc = gt[tr]
            if esamp < len(gs) and np.isfinite(gs[esamp]) and np.isfinite(gc[esamp]):  # FIX
                ph.append(np.arctan2(gs[esamp], gc[esamp]))
    if len(ph) < 2: return float('nan')
    return float(np.abs(np.mean(np.exp(1j * np.array(ph)))))

def alpha_snr_per_epoch(raw_1ch):
    snrs = []
    for tr in range(raw_1ch.shape[0]):
        ep = raw_1ch[tr].astype(np.float64)
        psd = np.abs(np.fft.rfft(ep)) ** 2
        freqs = np.fft.rfftfreq(len(ep), 1.0 / FS)
        mu = psd[(freqs >= 9) & (freqs <= 13)].mean()
        broad = psd[(freqs >= 1) & (freqs <= 40)].mean()
        snrs.append(mu / broad if broad > 0 else 0.0)
    return float(np.mean(snrs))

def estimate_alpha_peak(raw_1ch, onset_samp):
    seg = raw_1ch[:, onset_samp:]
    if seg.shape[1] < 16: return np.nan
    nfft = int(2 ** np.ceil(np.log2(seg.shape[1] * 4))); psd = None
    for tr in range(seg.shape[0]):
        f, px = welch(seg[tr].astype(np.float64), fs=FS, nperseg=seg.shape[1], nfft=nfft, detrend='linear')
        psd = px if psd is None else psd + px
    psd /= seg.shape[0]; band = (f >= ALPHA_SEARCH_LO) & (f <= ALPHA_SEARCH_HI)
    return float(f[band][np.argmax(psd[band])]) if np.any(band) else np.nan


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 20 subjects from /content/drive/MyDrive/RAIN_IRC/LEMON @ 1000 Hz

GT filter      : 401 taps (order 400, filtfilt) -- SCORING REFERENCE
PEAP internal  : 401 taps (order 400) -- fixed paper config
Descriptor     : 231 taps (order 230) -- EPN-comparable ITPC
PEAP AR: Burg order 130 | context 980ms | pad 290ms
Resampling: 1000 Hz -> 1000 Hz
GT requires epoch length > 1204 samples (1.20s) to run cleanly


In [ ]:
# =============================================================
# Main eval loop  (per-timepoint errors; EPN-parallel records)
# =============================================================
print(f"\n{'='*96}")
print("PEAP on Sensorimotor Mu/Alpha (Kirchhoff et al. 2026) -- FIR GT")
print(f"{'='*96}")
print(f"  Pipeline: raw -> Burg AR({PEAP_AR_ORDER}) -> pad {PEAP_PAD_MS}ms -> FIR({PEAP_INTERNAL_FIR_ORDER}) -> Hilbert")  # FIX: internal order
print(f"  Fs: {PEAP_FS:.0f} Hz | Context: {PEAP_CONTEXT_MS}ms | GT: zero-phase FIR order {GT_FIR_ORDER}")            # FIX: GT order
print(f"  Eval: {EVAL_TIMES_S} | Subjects: {len(all_subjects)} | No per-subject optimization (fixed paper params)")
print(f"\n{'Subject':<14s} {'N':>4s} {'MACE':>7s} {'CSD':>7s} {'Acc%':>6s} {'PLV':>6s} {'Bias':>7s} {'SNR':>5s} {'ITPC':>6s} {'Peak':>6s}")
print("-" * 86)

results_peap = []

for subj_id, subj_data in all_subjects.items():
    t0 = time.time()
    epochs_native = subj_data['epochs']
    n_epochs = subj_data['n_epochs']
    cal_n    = subj_data['cal_n']
    ch_name  = subj_data['montage']

    n_trials         = int(epochs_native.shape[0])
    epoch_len_native = epochs_native.shape[1]
    test_trials      = [int(tr) for tr in range(cal_n, n_epochs)]

    # --- alpha descriptors on NATIVE epochs, using the FIXED desc filter (EPN-comparable) ---
    alpha_snr = alpha_snr_per_epoch(epochs_native)
    gt_native = {tr: compute_desc_fir(epochs_native[tr].astype(np.float64)) for tr in test_trials}  # FIX: desc, not gt
    itpc_tp = {t: (itpc_from_gt(gt_native, test_trials, es) if es < epoch_len_native else float('nan'))
               for t, es in zip(EVAL_TIMES_S, EVAL_TIMES_SAMP_NATIVE)}
    itpc_mean = float(np.nanmean(list(itpc_tp.values())))
    f_peak    = estimate_alpha_peak(epochs_native, PRE_SAMP)

    # --- Resample all epochs to 1000 Hz for the PEAP pipeline ---
    epoch_len_1k = int(epoch_len_native * PEAP_FS / FS)
    epochs_1k = np.zeros((n_epochs, epoch_len_1k), dtype=np.float64)
    for tr in range(n_epochs):
        epochs_1k[tr] = resample(epochs_native[tr].astype(np.float64), epoch_len_1k)

    # FIX: warn once per subject if the tight GT can't run on this epoch length
    if epoch_len_1k < GT_MIN_LEN:
        print(f"  !! {subj_id}: epoch {epoch_len_1k} < GT_MIN_LEN {GT_MIN_LEN} -> GT will be NaN")

    # --- PEAP predictions on TEST trials, errors kept per eval timepoint ---
    per_tp = {t: [] for t in EVAL_TIMES_S}
    for tr in test_trials:
        gt_s, gt_c = compute_gt_fir(epochs_1k[tr])
        for i, es in enumerate(EVAL_TIMES_SAMP_1K):
            if es >= epoch_len_1k:
                continue
            ps, pc = peap_predict_phase(epochs_1k[tr], es)
            # FIX: skip if either estimate or GT is NaN at this sample
            if not (np.isfinite(ps) and np.isfinite(pc)
                    and np.isfinite(gt_s[es]) and np.isfinite(gt_c[es])):
                continue
            err = np.arctan2(ps * gt_c[es] - pc * gt_s[es],
                             pc * gt_c[es] + ps * gt_s[es])
            per_tp[EVAL_TIMES_S[i]].append(err)

    per_tp     = {t: np.array(e) for t, e in per_tp.items()}
    have_any   = any(len(per_tp[t]) for t in EVAL_TIMES_S)
    all_errs   = np.concatenate([per_tp[t] for t in EVAL_TIMES_S]) if have_any else np.array([])
    met        = errors_to_metrics(all_errs)
    per_tp_met = {t: errors_to_metrics(per_tp[t]) for t in EVAL_TIMES_S}

    # FIX: flag a dead subject loudly instead of printing a plausible 48%
    if met['n_samples'] == 0:
        print(f"  !! {subj_id}: 0 valid samples -- PEAP or GT returned all-NaN, check filters/epoch length")

    # --- per-call latency ---
    tsamps = []
    valid_es = [es for es in EVAL_TIMES_SAMP_1K if es < epoch_len_1k]
    if test_trials and valid_es:
        tr0, es0 = test_trials[0], valid_es[0]
        for _ in range(50):
            ts = time.perf_counter(); peap_predict_phase(epochs_1k[tr0], es0)
            tsamps.append((time.perf_counter() - ts) * 1000)

    dt = time.time() - t0
    pk = f"{f_peak:.2f}" if np.isfinite(f_peak) else " n/a"
    print(f"{subj_id:<14s} {met['n_samples']:>4d} {met['mace']:>6.1f}\u00b0 {met['csd']:>6.1f}\u00b0 "
          f"{met['acc']:>5.1f}% {met['plv']:>5.3f} {met['bias']:>+6.1f}\u00b0 "
          f"{alpha_snr:>5.2f} {itpc_mean:>6.3f} {pk:>6s}  [{dt:.0f}s]")

    results_peap.append({
        'name': subj_id, 'best_ch': ch_name, 'n_trials': n_trials,
        'n_test_samples': met['n_samples'], 'n_train_samples': 0,
        'alpha_snr': alpha_snr,
        'alpha_itpc_per_timepoint': {str(t): itpc_tp[t] for t in EVAL_TIMES_S},
        'alpha_itpc_mean': itpc_mean,
        'alpha_peak_hz': float(f_peak) if np.isfinite(f_peak) else None,
        'metrics_pooled': met,
        'metrics_per_timepoint': {str(t): per_tp_met[t] for t in EVAL_TIMES_S},
        'errors_per_timepoint_rad': {str(t): [float(x) for x in per_tp[t]] for t in EVAL_TIMES_S},
        'compute_time_ms': float(np.mean(tsamps)) if tsamps else 0.0})

    del epochs_1k, gt_native
    gc.collect()

print(f"\n{len(results_peap)} subjects done")


PEAP on Sensorimotor Mu/Alpha (Kirchhoff et al. 2026) -- FIR GT
  Pipeline: raw -> Burg AR(130) -> pad 290ms -> FIR(400) -> Hilbert
  Fs: 1000 Hz | Context: 980ms | GT: zero-phase FIR order 400
  Eval: [1.5, 2.0, 2.5, 3.0, 3.5] | Subjects: 20 | No per-subject optimization (fixed paper params)

Subject           N    MACE     CSD   Acc%    PLV    Bias   SNR   ITPC   Peak
--------------------------------------------------------------------------------------
sub-032311      405   28.6°   39.3°  84.1% 0.790   -0.6°  3.60  0.093  10.96  [11s]
sub-032317      410   32.7°   43.9°  81.9% 0.746   -2.1°  3.60  0.083  10.10  [11s]
sub-032324      400   25.2°   35.6°  86.0% 0.824   -3.2°  3.90  0.087  10.56  [11s]
sub-032330      430   32.5°   43.7°  81.9% 0.747   -4.8°  3.76  0.134  11.14  [11s]
sub-032332      400   29.3°   40.4°  83.7% 0.780   -0.8°  3.49  0.077  10.41  [11s]
sub-032355      430   31.2°   41.7°  82.7% 0.767   -1.4°  3.52  0.077  10.44  [11s]
sub-032361      405   25.6°   35.1°

In [ ]:
subj_id = 'sub-032311'
sd = all_subjects[subj_id]
epochs_native = sd['epochs']
n_epochs = sd['n_epochs']; cal_n = sd['cal_n']
epoch_len_native = epochs_native.shape[1]
epoch_len_1k = int(epoch_len_native * PEAP_FS / FS)

print(f"native epoch len : {epoch_len_native} samp @ {FS:.0f}Hz = {epoch_len_native/FS:.2f}s")
print(f"1k epoch len     : {epoch_len_1k} samp @ 1000Hz = {epoch_len_1k/1000:.2f}s")
print(f"GT_MIN_LEN       : {GT_MIN_LEN} samp ({GT_MIN_LEN/1000:.2f}s)")
print(f"GT runs?         : {epoch_len_1k >= GT_MIN_LEN}")
print(f"max EVAL sample  : {max(EVAL_TIMES_SAMP_1K)}  (epoch has {epoch_len_1k})")
print(f"eval past epoch? : {[es for es in EVAL_TIMES_SAMP_1K if es >= epoch_len_1k]}")

# test PEAP on one trial/timepoint
tr = cal_n; es = EVAL_TIMES_SAMP_1K[0]
epochs_1k0 = resample(epochs_native[tr].astype(np.float64), epoch_len_1k)
ps, pc = peap_predict_phase(epochs_1k0, es)
print(f"\nPEAP est at tr={tr}, es={es}: sin={ps}, cos={pc}  -> {'NaN' if not np.isfinite(ps) else 'OK'}")

# test GT on same trial
gt_s, gt_c = compute_gt_fir(epochs_1k0)
print(f"GT finite frac   : {np.isfinite(gt_s).mean():.2f}  (1.0 = all good, 0.0 = all NaN)")
print(f"GT at es={es}    : sin={gt_s[es] if es < len(gt_s) else 'OOB'}")

native epoch len : 5000 samp @ 1000Hz = 5.00s
1k epoch len     : 5000 samp @ 1000Hz = 5.00s
GT_MIN_LEN       : 1204 samp (1.20s)
GT runs?         : True
max EVAL sample  : 3500  (epoch has 5000)
eval past epoch? : []

PEAP est at tr=121, es=1500: sin=-0.6238595686489231, cos=-0.7815364601892734  -> OK
GT finite frac   : 1.00  (1.0 = all good, 0.0 = all NaN)
GT at es=1500    : sin=-0.9954719543457031


In [ ]:
tr = 121; es = 1500
epochs_1k0 = resample(epochs_native[tr].astype(np.float64), epoch_len_1k)

ctx_start = max(0, es - PEAP_CONTEXT_SAMP + 1)
context = epochs_1k0[ctx_start:es + 1].astype(np.float64)
print(f"context: len={len(context)}, std={np.std(context):.4g}, "
      f"finite={np.isfinite(context).all()}, max|x|={np.max(np.abs(context)):.4g}")

ar_order = min(PEAP_AR_ORDER, len(context) // 3)
coeffs = burg_ar(context, ar_order)
print(f"coeffs : order={ar_order}, finite={np.isfinite(coeffs).all()}, "
      f"max|c|={np.max(np.abs(coeffs)):.4g}")

# AR polynomial stability: roots of 1 - sum(c_k z^-k) inside unit circle?
roots = np.roots(np.concatenate([[1.0], -coeffs]))
print(f"AR roots: max|root|={np.max(np.abs(roots)):.4g}  ({'STABLE' if np.max(np.abs(roots))<1 else 'UNSTABLE -> diverges'})")

padded = ar_extrapolate(context, coeffs, PEAP_PAD_SAMP)
print(f"padded : len={len(padded)}, finite={np.isfinite(padded).all()}, "
      f"max|x|={np.nanmax(np.abs(padded)):.4g}")
print(f"  -> pad region finite? {np.isfinite(padded[len(context):]).all()}, "
      f"n_nan={np.isnan(padded).sum()}, n_inf={np.isinf(padded).sum()}")

filtered = filtfilt(peap_b, peap_a, padded, padlen=min(3*len(peap_b), len(padded)-1))
print(f"filtered: finite={np.isfinite(filtered).all()}, n_nan={np.isnan(filtered).sum()}")

context: len=980, std=1.976e-06, finite=True, max|x|=8.758e-06
coeffs : order=130, finite=True, max|c|=0.2265
AR roots: max|root|=0.9997  (STABLE)
padded : len=1270, finite=True, max|x|=8.758e-06
  -> pad region finite? True, n_nan=0, n_inf=0
filtered: finite=True, n_nan=0


In [ ]:
import numpy as np

def peap_debug(raw_signal_1k, eval_samp):
    ctx_start = max(0, eval_samp - PEAP_CONTEXT_SAMP + 1)
    context = raw_signal_1k[ctx_start:eval_samp + 1].astype(np.float64)
    ctx_len = len(context)
    print(f"[1] ctx_len={ctx_len}, std={np.std(context):.3g}")
    if ctx_len < PEAP_AR_ORDER + 5 or np.std(context) < 1e-10:
        print("    -> guard 1 (ctx_len/std) FIRED"); return np.nan, np.nan

    ar_order = min(PEAP_AR_ORDER, ctx_len // 3)
    coeffs = burg_ar(context, ar_order)
    padded = ar_extrapolate(context, coeffs, PEAP_PAD_SAMP)
    print(f"[2] padded len={len(padded)}, finite={np.isfinite(padded).all()}")

    min_filtfilt_len = 3 * len(peap_b) + 1
    print(f"[3] len(padded)={len(padded)} vs min_filtfilt_len={min_filtfilt_len}")
    if len(padded) < min_filtfilt_len:
        print("    -> guard 3 (filtfilt len) FIRED"); return np.nan, np.nan

    try:
        filtered = filtfilt(peap_b, peap_a, padded, padlen=min(3*len(peap_b), len(padded)-1))
    except Exception as e:
        print(f"    -> filtfilt EXCEPTION: {e}"); return np.nan, np.nan
    print(f"[4] filtered finite={np.isfinite(filtered).all()}")

    analytic = hilbert(filtered)
    target_idx = ctx_len - 1
    print(f"[5] target_idx={target_idx}, len(analytic)={len(analytic)}")
    if target_idx >= len(analytic):
        print("    -> guard 5 (target_idx OOB) FIRED"); return np.nan, np.nan

    ph = np.angle(analytic[target_idx])
    print(f"[6] ph={ph}, sin={np.sin(ph)}, cos={np.cos(ph)}")
    return np.sin(ph), np.cos(ph)

# run it exactly as the loop does
tr = 121; es = 1500
epochs_1k0 = resample(epochs_native[tr].astype(np.float64), epoch_len_1k)
print("peap_b length:", len(peap_b), " PEAP_INTERNAL_FIR_ORDER:", PEAP_INTERNAL_FIR_ORDER)
peap_debug(epochs_1k0, es)

peap_b length: 401  PEAP_INTERNAL_FIR_ORDER: 400
[1] ctx_len=980, std=1.98e-06
[2] padded len=1270, finite=True
[3] len(padded)=1270 vs min_filtfilt_len=1204
[4] filtered finite=True
[5] target_idx=979, len(analytic)=1270
[6] ph=-2.467921189103043, sin=-0.6238595686489231, cos=-0.7815364601892734


(np.float64(-0.6238595686489231), np.float64(-0.7815364601892734))

In [ ]:
# =============================================================
# SUMMARY + SAVE  (complete model-tagged JSON, EPN/BOB-parallel)
# =============================================================
valid = [r for r in results_peap if r['n_test_samples'] >= 10]
n = len(valid)
M   = np.array([r['metrics_pooled']['mace'] for r in valid])
P   = np.array([r['metrics_pooled']['plv'] for r in valid])
Cd  = np.array([r['metrics_pooled']['csd'] for r in valid])
A   = np.array([r['metrics_pooled']['acc'] for r in valid])
SNR = np.array([r['alpha_snr'] for r in valid])
IT  = np.array([r['alpha_itpc_mean'] for r in valid])

print('=' * 72); print('PEAP ALPHA (mu) — group summary'); print('=' * 72)
print(f"  n={n} participants (>=10 test samples)")
print(f"  MACE {M.mean():.1f}\u00b0\u00b1{M.std():.1f}   PLV {P.mean():.3f}   CSD {Cd.mean():.1f}\u00b0   Acc {A.mean():.1f}%")

print('\n--- Per-timepoint ---')
print(f"{'ms':>5}{'MACE':>9}{'PLV':>8}{'CSD':>8}{'Acc':>8}{'ITPC':>8}{'N':>7}")
for t in EVAL_TIMES_S:
    g = lambda k: np.array([r['metrics_per_timepoint'][str(t)][k] for r in valid])
    ti = np.array([r['alpha_itpc_per_timepoint'][str(t)] for r in valid])
    print(f"{int(t*1000):>5}{g('mace').mean():>8.1f}\u00b0{g('plv').mean():>8.3f}"
          f"{g('csd').mean():>7.1f}\u00b0{g('acc').mean():>7.1f}%{np.nanmean(ti):>8.3f}{g('n_samples').mean():>7.0f}")

group_stats = {'n_participants': int(n),
               'mean_mace': float(M.mean()), 'median_mace': float(np.median(M)),
               'mean_plv': float(P.mean()), 'mean_csd': float(Cd.mean()), 'mean_acc': float(A.mean())}

print('\n--- Accuracy correlations across subjects ---')
if n >= 3 and np.std(SNR) > 0:
    r1, p1 = pearsonr(SNR, A); rs1, ps1 = spearmanr(SNR, A)
    group_stats.update({'acc_vs_snr_pearson_r': float(r1), 'acc_vs_snr_pearson_p': float(p1),
                        'acc_vs_snr_spearman_r': float(rs1), 'acc_vs_snr_spearman_p': float(ps1)})
    print(f"  Acc vs SNR : r={r1:+.3f} (p={p1:.4f})  rho={rs1:+.3f} (p={ps1:.4f})")
if n >= 3 and np.std(IT) > 0:
    r2, p2 = pearsonr(IT, A); rs2, ps2 = spearmanr(IT, A)
    group_stats.update({'acc_vs_itpc_pearson_r': float(r2), 'acc_vs_itpc_pearson_p': float(p2),
                        'acc_vs_itpc_spearman_r': float(rs2), 'acc_vs_itpc_spearman_p': float(ps2)})
    print(f"  Acc vs ITPC: r={r2:+.3f} (p={p2:.4f})  rho={rs2:+.3f} (p={ps2:.4f})")

out = {'model': 'PEAP',
       'config': {'fs': FS, 'pre_samp': PRE_SAMP, 'eval_times_s': EVAL_TIMES_S, 'horizon_ms': 0.0,
                  'gt_band': [GT_LOW, GT_HIGH], 'gt_fir_numtaps': GT_FIR_NUMTAPS,
                  'peap_fs': PEAP_FS, 'ar_order': PEAP_AR_ORDER,
                  'context_ms': PEAP_CONTEXT_MS, 'context_samples': PEAP_CONTEXT_SAMP,
                  'pad_ms': PEAP_PAD_MS, 'pad_samples': PEAP_PAD_SAMP,
                  'fir_order': PEAP_FIR_ORDER, 'fir_band': PEAP_FIR_BAND,
                  'data_dir': LOAD_DIR, 'dataset': 'LEMON top-20 mu-SNR (C3 Hjorth)', 'band': 'mu/alpha 9-13 Hz',
                  'split': 'chronological cal/test (first cal_n epochs = cal)',
                  'channel': 'C3 Hjorth (single channel)', 'conditions': ['model'],
                  'itpc_basis': 'GT-phase ITPC, test trials, per eval point', 'errors_unit': 'radians',
                  'no_training': True,
                  'note': ('single-condition method (fixed paper params, no per-subject training -> '
                           'n_train_samples=0); headline metrics under metrics_pooled / '
                           'metrics_per_timepoint / errors_per_timepoint_rad (analogue of BOB opt_*). '
                           'alpha_snr / alpha_itpc_* / alpha_peak_hz are computed identically to EPN on the '
                           'native epochs + shared FIR-231 GT, so they match per subject across models. '
                           'Same data/GT/split as EPN/BOB LEMON -> join on name.')},
       'group_stats': group_stats,
       'subjects': results_peap}
outpath = os.path.join(OUTDIR, 'peap_alpha_results.json')
with open(outpath, 'w') as f:
    json.dump(out, f, indent=2, default=str)
print(f"\nSaved -> {outpath}")
print("Per subject: metrics_pooled + per-timepoint metrics, per-timepoint errors (rad), SNR, ITPC, peak.")
print("Same data/GT/split as EPN/BOB LEMON -> join on 'name';"
      " PEAP.metrics_pooled <-> EPN.metrics_pooled <-> BOB.opt_metrics_pooled.")


PEAP ALPHA (mu) — group summary
  n=20 participants (>=10 test samples)
  MACE 28.4°±4.2   PLV 0.795   CSD 38.6°   Acc 84.2%

--- Per-timepoint ---
   ms     MACE     PLV     CSD     Acc    ITPC      N
 1500    29.3°   0.785   39.6°   83.7%   0.081     84
 2000    28.8°   0.790   39.0°   84.0%   0.113     84
 2500    27.9°   0.805   37.4°   84.5%   0.086     84
 3000    27.9°   0.798   38.1°   84.5%   0.102     84
 3500    28.2°   0.801   37.9°   84.3%   0.104     84

--- Accuracy correlations across subjects ---
  Acc vs SNR : r=+0.628 (p=0.0030)  rho=+0.340 (p=0.1426)
  Acc vs ITPC: r=-0.116 (p=0.6257)  rho=-0.099 (p=0.6772)

Saved -> /content/drive/MyDrive/RAIN_IRC/result_alpha/PEAP_lemon/peap_alpha_results.json
Per subject: metrics_pooled + per-timepoint metrics, per-timepoint errors (rad), SNR, ITPC, peak.
Same data/GT/split as EPN/BOB LEMON -> join on 'name'; PEAP.metrics_pooled <-> EPN.metrics_pooled <-> BOB.opt_metrics_pooled.
